# লেসন ১২ - এজেন্ট স্ক্র্যাপ্যাড সহ চ্যাট ইতিহাস হ্রাস

এই নোটবুকটি দেখায় কীভাবে Microsoft Agent Framework ব্যবহার করে দীর্ঘ আলাপচারিতায় প্রসঙ্গ পরিচালনা করা যায়। আলাপচারিতা যত বাড়ে, টোকেনের সংখ্যা বাড়ে — শেষ পর্যন্ত মডেলের প্রসঙ্গ জানালার সীমা ছাড়িয়ে যায়। আমরা এটি সমাধান করি একটি **প্রসঙ্গ সারসংক্ষেপ প্যাটার্ন** এবং একটি **এজেন্ট স্ক্র্যাপ্যাড** দিয়ে যা স্থায়ী স্মৃতি প্রদান করে।

## আপনি যা শিখবেন:
1. **কেন প্রসঙ্গ পরিচালনা গুরুত্বপূর্ণ**: টোকেন সীমা এবং প্রসঙ্গ জানালা বোঝা
2. **প্রসঙ্গ সচেতন এজেন্ট**: এমন এজেন্ট তৈরি করা যারা নিজের আলাপচারিতার প্রসঙ্গ পরিচালনা করে
3. **প্রসঙ্গ সারসংক্ষেপ প্যাটার্ন**: আলাপচারিতার ইতিহাস সংক্ষিপ্ত করার জন্য সরঞ্জাম ব্যবহার
4. **এজেন্ট স্ক্র্যাপ্যাড**: প্রসঙ্গ হ্রাসের পরেও থাকা স্থায়ী স্মৃতি

## পূর্বশর্ত:
- পরিবেশ ভেরিয়েবল গুলো সেট করা সহ Azure OpenAI সেটআপ
- পূর্ববর্তী লেসন থেকে মৌলিক এজেন্ট ধারণাগুলি বোঝা


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## কেন প্রসঙ্গ পরিচালনা গুরুত্বপূর্ণ

প্রত্যেক LLM-এর একটি সীমিত **প্রসঙ্গ উইন্ডো** থাকে — এককে একটি অনুরোধে সর্বোচ্চ কতগুলি টোকেন প্রক্রিয়াকরণ করতে পারে। একটি বহুমুখী কথোপকথন এগিয়ে গেলে:

- প্রতিটি ব্যবহারকারী বার্তা এবং সহকারী प्रतिक्रিয়ার সাথে **টোকেন সংখ্যা সরলরেখাগত বৃদ্ধি পায়**।
- **প্রম্পট টোকেন ব্যয়কে প্রাধান্য দেয়** কারণ সম্পূর্ণ ইতিহাস প্রতিটি পর্যায়ে পুনরায় পাঠানো হয়।
- অবশেষে কথোপকথন **প্রসঙ্গ উইন্ডো ছাড়িয়ে যায়** এবং মডেল হয় অনুচ্ছেদ কেটে ফেলে বা ত্রুটি দেয়।

### প্রসঙ্গ পরিচালনার কৌশলসমূহ

| কৌশল | কীভাবে কাজ করে | সুবিধা-অসুবিধা |
|---|---|---|
| **কাটাছেঁড়া** | সবচেয়ে পুরানো বার্তাগুলো বাদ দেয় | প্রাথমিক প্রসঙ্গ হারায় |
| **সারাংশ তৈরি** | পুরনো বার্তাগুলোকে একটি সারাংশে রূপান্তরিত করে | কিছু বিস্তারিত হারায়, কিন্তু মূল বিষয়গুলো বজায় থাকে |
| **স্ক্র্যাচপ্যাড / বাহ্যিক মেমরি** | কথোপকথনের বাইরে গুরুত্বপূর্ণ তথ্য সংরক্ষণ করে | টুল কলের প্রয়োজন হয়, কিন্তু কোনও সংকোচন সহ্য করে |

এই নোটবুকে আমরা **সারাংশ তৈরি** এবং একটি **স্ক্র্যাচপ্যাড টুল** একত্রিত করেছি যাতে এজেন্ট কথোপকথনের পূর্বাবস্থাকে বজায় রাখতে পারে এমনকি যখন ইতিহাস সংক্ষেপিত হয়।


## একটি প্রেক্ষাপট-সচেতন এজেন্ট তৈরি করা


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## দীর্ঘ কথোপকথনের সিমুলেশন

চলুন একটি বহু-টর্ন কথোপকথনের মাধ্যমে দেখা যাক কীভাবে প্রসঙ্গ জমা হয়। এজেন্টকে টার্নগুলোর মধ্যে গুরুত্বপূর্ণ বিবরণ (পছন্দ, বাজেট, ভ্রমণের তারিখ) ধরে রাখতে হবে এবং ধারাবাহিকতা প্রদর্শন করতে হবে।


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

লক্ষ্য করুন কীভাবে এজেন্ট পূর্ববর্তী কথোপকথন থেকে প্রসঙ্গ ধারণ করে — এটি জাপান, সুশি, মন্দির, ফটোগ্রাফি, $3000 বাজেট, একক ভ্রমণ এবং এপ্রিল সময়সীমা সম্পর্কে জানে। সংক্ষিপ্ত একটি কথোপকথনে এটি ভাল কাজ করে, কিন্তু কথোপকথন যত বাড়ে পুরো ইতিহাস পুনরায় পাঠানো ব্যয়বহুল হয়ে ওঠে।

চলুন আরও কয়েকটি পর্যায়ের মাধ্যমে কথোপকথন চালিয়ে যাই প্রসঙ্গ সঞ্চয়ের জন্য:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## প্রেক্ষাপট সংক্ষেপণ প্যাটার্ন

যখন কথোপকথন বাড়ে, তখন আমরা একটি **সংক্ষেপণ সরঞ্জাম** ব্যবহার করতে পারি সংগৃহীত প্রেক্ষাপটকে একটি সংক্ষিপ্ত বিন্যাসে সংকুচিত করার জন্য। এজেন্ট এই সরঞ্জামটি কল করে মূল পছন্দগুলিকে রেকর্ড করার জন্য যাতে পুরাতন বার্তাগুলি বাদ দেওয়া হলেও মূল তথ্য সংরক্ষিত থাকে।

এই প্যাটার্নটি আরও উন্নত ইতিহাস হ্রাসের জন্য বিল্ডিং ব্লক:
1. এজেন্ট কথোপকথন থেকে গুরুত্বপূর্ণ তথ্য শনাক্ত করে
2. এটি সংক্ষেপণের সরঞ্জামটি কল করে সেগুলো সংরক্ষণ করে
3. পুরনো বার্তাগুলো নিরাপদে মুছে ফেলা যায় কারণ সংক্ষেপণে যা গুরুত্বপূর্ণ তা ধারণ করা হয়

নিচে আমরা একটি `summarize_preferences` সরঞ্জাম সংজ্ঞায়িত করেছি যা এজেন্ট কল করতে পারে শিখে নেওয়া তথ্যের একটি সংক্ষিপ্ত সারাংশ রেকর্ড করার জন্য।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## সারসংক্ষেপ

এই পাঠে আপনি শিখেছেন কিভাবে Microsoft Agent Framework ব্যবহার করে দীর্ঘকালীন এজেন্ট কথোপকথনে প্রসঙ্গ পরিচালনা করতে হয়:

### মূল ধারণা
- **কন্টেক্সট উইন্ডোগুলো সীমিত** — কথোপকথন ইতিহাসের প্রতিটি টোকেনের জন্য খরচ হয় এবং এটি সীমার মধ্যে গণ্য হয়।
- **সারাংশ তৈরির সরঞ্জাম** এজেন্টকে সঞ্চিত প্রসঙ্গ компакт সারাংশে রূপান্তর করতে দেয়, টোকেন ব্যবহার কমায় এবং অপরিহার্য তথ্য সংরক্ষণ করে।
- **এজেন্ট স্ক্র্যাচপ্যাড** একটি স্থায়ী বাহ্যিক স্মৃতি প্রদান করে যা কোনও কথোপকথন হ্রাসের পরও টিকে থাকে।

### আপনি যা তৈরি করলেন
- একটি **কন্টেক্সট-সচেতন এজেন্ট** যা বহু-বর্তী কথোপকথনে ধারাবাহিকতা বজায় রাখে
- একটি **সারাংশ তৈরির সরঞ্জাম** (`summarize_preferences`) যা সংক্ষিপ্ত ফরম্যাটে গুরুত্বপূর্ণ ব্যবহারকারীর তথ্য রেকর্ড করে
- একটি **বহু-বর্তী কথোপকথন** যা প্রসঙ্গ রক্ষণ এবং পরিবর্তন পরিচালনা প্রদর্শন করে

### বাস্তব জীবনের প্রয়োগসমূহ
- **কাস্টমার সার্ভিস বটস**: দীর্ঘকালীন সহায়তা সেশনের সময় পছন্দ স্মরণ করে
- **ব্যক্তিগত সহকারী**: চলমান প্রকল্প ট্র্যাক করে পুনরায় প্রসঙ্গ ব্যাখ্যার প্রয়োজন ছাড়াই
- **শিক্ষামূলক টিউটরস**: অনেক আলাপচারিতায় ছাত্রের অগ্রগতি রক্ষা করে

### পরবর্তী ধাপ
- ফাইল-ভিত্তিক স্থায়ীত্বের সাথে সম্পূর্ণ স্ক্র্যাচপ্যাড সরঞ্জাম বাস্তবায়ন করুন
- সারাংশ তৈরি হওয়ার পর স্বয়ংক্রিয় ইতিহাস সংক্ষিপ্তকরণ যোগ করুন
- সেম্যান্টিক মেমোরি অনুসন্ধানের জন্য ভেক্টর ডাটাবেসের সাথে সংযুক্ত করুন
- এমন এজেন্ট তৈরি করুন যা সম্পূর্ণ প্রসঙ্গ সহ দিন পর দিন কথোপকথন পুনরায় শুরু করতে পারে


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:  
এই দলিলটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা সঠিকতার জন্য চেষ্টা করি, তবে অনুগ্রহ করে জানুন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসম্পূর্ণতা থাকতে পারে। মূল নথির আদি ভাষায় থাকা দলিলকে কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানুষের অনুবাদ নেওয়া প্রয়োজন। এই অনুবাদের ব্যবহারের ফলে যে কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
